In [1]:
import pandas as pd
import ccf_streamlines.morphology as ccfmorph
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

import numpy as np
import json
import seaborn as sns
import umap
import os 

c:\Users\sarah.wallingbell\AppData\Local\anaconda3\envs\morph_utils_v3\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# out_dir = r'\\allen\programs\celltypes\workgroups\mousecelltypes\SarahWB\datasets\human_exc\analysis\20250815_seaad\MET_typing\config' #V2 4000 genes
out_dir = r'\\allen\programs\celltypes\workgroups\mousecelltypes\SarahWB\datasets\human_exc\analysis\20250815_seaad\MET_typing\config_v2_binary' #V2 1000 genes

Ephys & Morph cell-cell correlations

In [3]:
data_root = r'\\allen\programs\celltypes\workgroups\mousecelltypes\SarahWB\datasets\human_exc\analysis\20250815_seaad\MET_typing\data' 

ephys_df = pd.read_csv(os.path.join(data_root, 'e_data.csv'), index_col=0)
morph_df = pd.read_csv(os.path.join(data_root, 'm_data.csv'), index_col=0)

ephys_corr_df = pd.DataFrame(np.corrcoef(ephys_df.values), index=ephys_df.index, columns=ephys_df.index)
morph_corr_df = pd.DataFrame(np.corrcoef(morph_df), index=morph_df.index, columns=morph_df.index)

ephys_corr_df.to_csv(os.path.join(out_dir, 'met_corr_ephys.csv'))
morph_corr_df.to_csv(os.path.join(out_dir, 'met_corr_morph.csv'))

Transcriptomic cell-cell correlations

In [4]:
anno_df = pd.read_feather(os.path.join(data_root, 't_data.feather'))
anno_df["spec_id_label"] = pd.to_numeric(anno_df["spec_id_label"])

color_dict = dict(zip(anno_df["cluster_label"], anno_df["cluster_color"]))

In [5]:
#NOTE: we need to run ME clustering before these will exist!
me_types = pd.read_csv(os.path.join(out_dir, 'refined_text_labels.csv'), index_col=0)
unrefined_me_types = pd.read_csv(os.path.join(out_dir, 'clust_labels.csv'), index_col=0)


In [6]:
# selection of high variance genes by Jeremy 
tx_root = r'\\allen\programs\celltypes\workgroups\mousecelltypes\SarahWB\datasets\human_exc\data\transcriptomics\20251003'

#NOTE THIS IS THE THING TO CHANGE between the 4000 and 1000 gene runs of this V2 MET typing
# exc_de_genes = pd.read_csv(os.path.join(tx_root, 'ref_all_exc_de_genes.csv'), index_col=0) #4000 genes
exc_de_genes = pd.read_csv(os.path.join(tx_root, 'binary_genes.csv'), index_col=0) #1000 genes

exc_select_marker_genes = exc_de_genes["x"].values

In [7]:
#load standard Patch-seq gene expression data
genes_df = pd.read_feather(os.path.join(tx_root, 'human_cpm_samp_dat_rsc392.feather'), columns=["exp_component_name"] + exc_select_marker_genes.tolist())
genes_df = genes_df.rename(columns={'exp_component_name' : 'sample_id'})
genes_df.set_index("sample_id", inplace=True)
genes_df

,CAMTA1,KAZN,IGSF21,RAP1GAP,CSMD2,GRIK3,HIVEP3,SLC2A1,RNF220,AGBL4,...,IL1RAPL2,PAK3,HTR2C,DANT2,GRIA3,TENM1,AL008633.1,FGF13,LINC00632,IDS
sample_id,,,,,,,,,,,,,,,,,,,,,
AB-S40301_S284_E1-50,0.000000,105.977109,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,105.977109,0.000000,...,0.000000,105.977109,0.000000,0.000000,0.000000,105.977109,211.954218,0.000000,0.000000,0.000000
AB-S40301_S285_E1-50,39.216840,29.412630,98.042099,0.000000,0.000000,176.475779,0.000000,0.000000,0.000000,794.141004,...,0.000000,0.000000,0.000000,0.000000,9.804210,137.258939,0.000000,0.000000,19.608420,0.000000
AB-S40301_S288_E1-50,200.873476,534.582637,32.398948,149.035159,155.514949,6.479790,217.072949,0.000000,6.479790,644.739059,...,9.719684,586.420953,3.239895,563.741690,411.466636,3.239895,592.900743,145.795265,2125.370968,213.833055
AB-S40301_S404_E1-50,361.622134,872.485148,154.980914,51.660305,0.000000,0.000000,172.201016,0.000000,177.941050,1366.128060,...,97.580576,269.781592,0.000000,34.440203,51.660305,0.000000,22.960135,109.060643,11.480068,45.920271
AB-S40301_S405_E1-50,457.282219,1871.469824,0.000000,59.277325,101.618271,1100.864602,618.177815,0.000000,194.768353,355.663948,...,127.022839,262.513867,0.000000,0.000000,8.468189,186.300163,0.000000,0.000000,440.345841,67.745514
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
SM-LNJMY_S423_E1-50,370.027024,3099.960439,0.000000,120.061960,423.169203,13.777602,1206.524285,1.968229,23.618746,173.204139,...,13.777602,149.585393,1.968229,251.933293,320.821303,33.459890,11.809373,1129.763360,206.664029,141.712477
SM-LNJMY_S424_E1-50,203.438278,1730.080144,0.000000,20.514784,415.424383,0.000000,0.000000,107.702618,1.709565,3.419131,...,10.257392,112.831314,3.419131,100.864356,143.603490,138.474794,61.544353,3.419131,165.827840,0.000000
SM-LNJMY_S425_E1-50,147.239264,117.791411,0.000000,343.558282,186.503067,0.000000,0.000000,0.000000,9.815951,0.000000,...,0.000000,68.711656,29.447853,0.000000,9.815951,873.619632,480.981595,107.975460,0.000000,0.000000


In [8]:
met_sample_ids = anno_df.set_index("spec_id_label").loc[me_types.index, "sample_id"].values
genes_met_df = genes_df.loc[met_sample_ids, :].copy()

genes_corr_df = pd.DataFrame(np.corrcoef(genes_met_df.apply(np.log1p)), index=me_types.index, columns=me_types.index)


In [9]:
genes_met_df

,CAMTA1,KAZN,IGSF21,RAP1GAP,CSMD2,GRIK3,HIVEP3,SLC2A1,RNF220,AGBL4,...,IL1RAPL2,PAK3,HTR2C,DANT2,GRIA3,TENM1,AL008633.1,FGF13,LINC00632,IDS
sample_id,,,,,,,,,,,,,,,,,,,,,
AB-S40380_S438_E1-50,510.525842,1520.657186,54.601694,0.000000,2.730085,218.406777,305.769488,0.000000,10.920339,734.392788,...,417.702961,2.730085,428.623300,0.000000,2.730085,0.000000,565.127536,0.000000,114.663558,0.000000
AB-S40358_S045_E1-50,470.505509,2016.452182,113.748585,113.748585,248.178730,0.000000,82.726243,0.000000,10.340780,1256.404821,...,811.751263,475.675899,0.000000,98.237414,423.971997,0.000000,351.586534,5.170390,1142.656236,201.645218
AB-S40353_S711_E1-50,583.411121,1019.580388,38.894075,25.003334,175.023336,183.357781,155.576299,16.668889,5.556296,247.255190,...,33.337778,455.616304,2.778148,102.791483,286.149264,163.910744,102.791483,25.003334,327.821487,147.241854
AB-S40352_S173_E1-50,468.450607,99.734645,117.868217,27.200358,208.536077,12.089048,178.313457,6.044524,21.155834,211.558339,...,24.178096,293.159412,0.000000,81.601074,24.178096,0.000000,226.669649,129.957265,129.957265,139.024051
AB-S40349_S064_E1-50,525.152018,891.376451,41.459370,421.503593,686.382900,0.000000,204.993551,0.000000,36.852773,142.804496,...,105.951723,635.710337,598.857564,0.000000,13.819790,0.000000,308.641975,4.606597,838.400590,110.558320
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
huPSX041217_E1-50_S01,485.289450,1339.835711,93.508656,77.127578,238.208183,0.000000,88.730842,0.000000,0.000000,2452.383959,...,2.047635,589.036281,0.000000,623.846072,586.306101,0.000000,485.971995,41.635241,1777.347016,174.731504
huPSX170327_E1-50_S42,339.738243,472.742150,3.614237,30.359588,73.730427,0.000000,160.472106,1.445695,10.119863,799.469141,...,1.445695,112.764183,0.000000,268.899205,302.873029,399.734570,308.655808,46.262229,1006.926323,116.378419
huPSX170327_E1-50_S41,921.787687,420.535363,48.430176,28.250936,317.217654,0.000000,125.918458,0.000000,0.000000,154.169394,...,3.228678,545.646652,0.807170,59.730551,120.268271,443.136112,898.379768,183.227500,1883.126684,918.559009


In [10]:
genes_corr_df.to_csv(os.path.join(out_dir, 'met_corr_genes.csv'))